In [1]:
import amy
import random
import time
import threading
import ipywidgets as W
from IPython.display import display

### Setup amy

In [2]:
amy.live(default_synths=1, audio_in=0)
amy.reset()

### Use Oscillator

In [3]:
freq = W.FloatSlider(value=220, min=50, max=1000, step=1, description="Freq")

btn_on = W.Button(description="On", button_style="success")
btn_off = W.Button(description="Off", button_style="danger")
playing = {"on": False}

def play(_):
    f = freq.value
    amy.send(osc=0, wave=amy.SAW_DOWN, freq=f, vel=1)
    amy.send(osc=1, wave=amy.SAW_DOWN, freq=f*2, vel=1)

def stop(_):
    amy.send(osc=0, vel=0)
    amy.send(osc=1, vel=0)

def update(_):
    if playing["on"]:
        f = freq.value
        amy.send(osc=0, freq=f)
        amy.send(osc=1, freq=f*2)

btn_on.on_click(play)
btn_off.on_click(stop)
freq.observe(update, names="value")

display(W.VBox([freq, W.HBox([btn_on, btn_off])]))

### Dynamic sequence

In [15]:
running = {"on": False}
big_fx = {"on": False}

BASE   = W.IntSlider(value=45, min=36, max=60, step=1, description="Base")
STEP   = W.FloatSlider(value=0.22, min=0.12, max=0.45, step=0.01, description="Step")
CHANGE = W.FloatSlider(value=0.20, min=0.0, max=1.0, step=0.05, description="Change")

midi_to_freq = lambda n: 440.0 * 2 ** ((n - 69) / 12)

def all_off():
    for osc in range(16):
        amy.send(osc=osc, vel=0)

def play_note(midi, accent=1.0, cutoff=950):
    f = midi_to_freq(midi)
    amy.send(
        osc=0, wave=amy.SAW_DOWN, freq=f*0.998, vel=0.56*accent,
        bp0="0,0,18,1,180,0.78,900,0",
        eg0_type=amy.ENVELOPE_TRUE_EXPONENTIAL,
    )
    amy.send(
        osc=1, wave=amy.SAW_DOWN, freq=f*1.002, vel=0.56*accent,
        bp0="0,0,22,1,220,0.72,1000,0",
        eg0_type=amy.ENVELOPE_TRUE_EXPONENTIAL,
    )
    amy.send(
        osc=2, wave=amy.PULSE, freq=f/2, vel=0.24*accent,
        bp0="0,0,30,1,260,0",
        eg0_type=amy.ENVELOPE_TRUE_EXPONENTIAL,
    )

def maybe_variant(pattern, p):
    out = pattern[:]
    if random.random() < p:
        i = random.randrange(len(out))
        out[i] = random.choice([0, 3, 5, 7, 10, 12])
    if random.random() < p * 0.6:
        out[-1] += 12
    return out

def loop():
    base_patterns = [
        [0, 3, 7, 10, 12, 10, 7, 3],
        [0, 3, 7, 12, 10, 7, 3, 0],
        [0, 5, 7, 10, 12, 10, 7, 5],
    ]
    accents = [1.15, 0.9, 1.0, 0.92, 1.08, 0.9, 1.0, 0.95]
    pattern = base_patterns[0]
    bar = 0

    while running["on"]:
        if bar % 4 == 0:
            base = random.choice(base_patterns)
            pattern = maybe_variant(base, CHANGE.value)

        for i, interval in enumerate(pattern):
            if not running["on"]:
                break

            note = BASE.value + interval
            cutoff = 820 + i * 50 + (20 if big_fx["on"] else 0)
            play_note(note, accent=accents[i], cutoff=cutoff)

            if big_fx["on"] and random.random() < CHANGE.value * 0.35 and i in (3, 7):
                play_note(note + 12, accent=0.55, cutoff=cutoff * 1.1)

            time.sleep(STEP.value)

        bar += 1

def start(_):
    if running["on"]:
        return
    amy.reset()
    running["on"] = True
    threading.Thread(target=loop, daemon=True).start()

def stop(_):
    running["on"] = False
    all_off()



start_btn = W.Button(description="Start", button_style="success")
stop_btn  = W.Button(description="Stop", button_style="danger")

start_btn.on_click(start)
stop_btn.on_click(stop)

display(W.VBox([
    BASE, STEP, CHANGE,
    W.HBox([start_btn, stop_btn])
]))

### Filter

In [12]:
cutoff = W.FloatSlider(value=1200, min=50, max=12000, step=50, description="Cutoff")
res = W.FloatSlider(value=0.8, min=0.5, max=16.0, step=0.1, description="Res")

btn_on = W.Button(description="Filter On", button_style="success")
btn_off = W.Button(description="Filter Off", button_style="danger")

NUM_OSC = 64
filter_on = {"state": True}


def apply_filter(_=None):
    if not filter_on["state"]:
        return
    for osc in range(NUM_OSC):
        amy.send(
            osc=osc,
            filter_type=amy.FILTER_LPF,
            filter_freq=cutoff.value,
            resonance=res.value,
        )


def enable_filter(_):
    filter_on["state"] = True
    apply_filter()


def disable_filter(_):
    filter_on["state"] = False
    for osc in range(NUM_OSC):
        amy.send(osc=osc, filter_type=0)  # bypass


cutoff.observe(apply_filter, names="value")
res.observe(apply_filter, names="value")

btn_on.on_click(enable_filter)
btn_off.on_click(disable_filter)

display(W.VBox([
    cutoff,
    res,
    W.HBox([btn_on, btn_off])
]))

### Reverb

In [13]:
rev_level = W.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.01, description="Level")
rev_live  = W.FloatSlider(value=0.85, min=0.0, max=1.0, step=0.01, description="Liveness")
rev_damp  = W.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01, description="Damping")
rev_xover = W.FloatSlider(value=3000, min=100, max=12000, step=100, description="Xover Hz")

btn_rev_on = W.Button(description="Reverb On", button_style="success")
btn_rev_off = W.Button(description="Reverb Off", button_style="danger")

reverb_on = {"state": True}


def apply_reverb(_=None):
    if not reverb_on["state"]:
        return
    amy.send(
        reverb=[
            rev_level.value,
            rev_live.value,
            rev_damp.value,
            rev_xover.value,
        ]
    )


def enable_reverb(_):
    reverb_on["state"] = True
    apply_reverb()


def disable_reverb(_):
    reverb_on["state"] = False
    amy.send(reverb=[0, rev_live.value, rev_damp.value, rev_xover.value])


rev_level.observe(apply_reverb, names="value")
rev_live.observe(apply_reverb, names="value")
rev_damp.observe(apply_reverb, names="value")
rev_xover.observe(apply_reverb, names="value")

btn_rev_on.on_click(enable_reverb)
btn_rev_off.on_click(disable_reverb)

display(W.VBox([
    rev_level,
    rev_live,
    rev_damp,
    rev_xover,
    W.HBox([btn_rev_on, btn_rev_off])
]))

### Envelope

In [14]:
attack = W.FloatSlider(value=50, min=0, max=5000, step=1, description="Attack ms")
decay = W.FloatSlider(value=200, min=0, max=5000, step=1, description="Decay ms")
sustain = W.FloatSlider(value=0.7, min=0.0, max=1.0, step=0.01, description="Sustain")
release = W.FloatSlider(value=300, min=0, max=8000, step=1, description="Release ms")

btn_env_on = W.Button(description="Env On", button_style="success")
btn_env_off = W.Button(description="Env Off", button_style="danger")

NUM_OSC = 64
env_on = {"state": True}


def env_bp0():
    return f"{attack.value},1,{decay.value},{sustain.value},{release.value},0"


def apply_env(_=None):
    if not env_on["state"]:
        return
    for osc in range(NUM_OSC):
        amy.send(
            osc=osc,
            bp0=env_bp0(),
        )


def enable_env(_):
    env_on["state"] = True
    apply_env()


def disable_env(_):
    env_on["state"] = False
    for osc in range(NUM_OSC):
        amy.send(
            osc=osc,
            bp0="0,1,0,1,0,0"
        )


attack.observe(apply_env, names="value")
decay.observe(apply_env, names="value")
sustain.observe(apply_env, names="value")
release.observe(apply_env, names="value")

btn_env_on.on_click(enable_env)
btn_env_off.on_click(disable_env)

display(W.VBox([
    attack,
    decay,
    sustain,
    release,
    W.HBox([btn_env_on, btn_env_off])
]))